In [2]:
import numpy as np
import lightgbm as lgb
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [3]:
X, y = make_classification(n_samples= 10000, n_features= 20, random_state= 42)
X_train , X_test, y_train, y_test = train_test_split(X, y,test_size= 0.3, random_state= 42, stratify=y)

In [4]:
# Dataset in LightGBM.

# LightGBM does not train on raw data. It discretizes continuous features into histogram bins, tries to combine categorical features, and automatically handles missing and infinite values
# 2. The Critical Step: Create Dataset Object (for speed)
#    LightGBM has its own internal data format that pre-bins the data.
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [13]:
parameters = {
    'objective' : 'binary',
    'metric' : 'binary_logloss',
    'boosting_type' : 'gbdt',
    'num_leaves' : 31,
    'learning_rate' : 0.05,
    'feature_fraction': 0.9,
    'verbose' : 0 # Shut up the training output for now
}
model = lgb.train(
    parameters,
    train_data,
    valid_sets = [test_data],
    num_boost_round = 100,
    callbacks = [lgb.early_stopping(stopping_rounds= 10)]
    # Stop if no improvement
)

y_pred = model.predict(X_test)
y_pred[:5]



Training until validation scores don't improve for 10 rounds
Early stopping, best iteration is:
[90]	valid_0's binary_logloss: 0.189853


array([0.86553475, 0.00743043, 0.98416751, 0.00752782, 0.98000416])

In [11]:
## array([0.86553475, 0.00743043, 0.98416751, 0.00752782, 0.98000416])
# this numbers define probability of correct classification

In [16]:
## for full report
y_pred_class = (y_pred > 0.5).astype(int)
from sklearn.metrics import classification_report
report = classification_report(y_test, y_pred_class)
from IPython.display import display, Markdown
display(Markdown(report))


              precision    recall  f1-score   support

           0       0.95      0.92      0.94      1502
           1       0.93      0.95      0.94      1498

    accuracy                           0.94      3000
   macro avg       0.94      0.94      0.94      3000
weighted avg       0.94      0.94      0.94      3000
